# Stage 05 — H3 Statistical Repair Only

## Scope

This notebook performs a **surgical recomputation of the H3 statistical layer only** from immutable Stage-05 prediction artifacts.

It does not train models, fit calibration, run transformer inference, alter thresholds, modify cue definitions, change dataset roles, or recompute H1/H2/H3-P/H4.

### Required mounted input

A completed Stage-05 output directory containing:

- `LOCKED_SOURCE_TEST_PREDICTIONS.parquet`
- `BANGLACLICK_PLATFORM_PREDICTIONS.parquet`
- `BAITBUSTER_HUMAN_PREDICTIONS.parquet`
- `H3_RESULTS.json` from the original successful run
- `STATISTICAL_SUMMARY.json`
- `STAGE05_ARTIFACT_MANIFEST.json`
- `STAGE05_SHA256SUMS.txt`
- `STAGE_STATUS.json`

The directory may be mounted anywhere under `/kaggle/input`, `/kaggle/working`, or the current working directory. Identical duplicate mounts are collapsed; conflicting scientific identities hard-stop.

### Confirmatory estimability rule

The locked bootstrap remains an ordinary cluster bootstrap: for each domain with `K` observed clusters, each replicate samples exactly `K` clusters uniformly with replacement.

A cue-transport statistic is undefined if a sampled source or target contains zero total multiplicity in either the cue-present or cue-absent group. Such a replicate is explicitly counted as invalid.

Because the locked protocol did not prespecify rejection-resampling or conditioning on valid bootstrap draws, confirmatory percentile CIs and bootstrap p-values are reported only when **all 5,000 locked replicates are estimable**. Otherwise the contrast is reported `NON_ESTIMABLE_UNDER_LOCKED_BOOTSTRAP`, with its observed point estimate retained but confirmatory CI/p set to null.

The global H3 outcome test remains the prespecified **32-dimensional** BanglaClick test. No dimension is dropped post hoc. If any of its 32 point components is undefined, any bootstrap element is undefined, or the complete 32×32 covariance is rank-deficient, the full global test is reported `NON_ESTIMABLE`. No reduced test is substituted.

BH-FDR is not run on a post-hoc subset of cue families. Family-level Simes p-values may be reported where all four prespecified platform contrasts are estimable, but the eight-family BH procedure is confirmatory only if all eight family p-values are available.


In [1]:
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import copy, hashlib, json, math, os, platform, re, sys, time
import numpy as np
import pandas as pd
from scipy.stats import chi2
from statsmodels.stats.multitest import multipletests

STAGE_ID = "05-H3-REPAIR"
REPAIR_VERSION = "SWARABYANJAN-H3-ESTIMABILITY-REPAIR-v1.0.0"
BOOTSTRAP_REPS = 5000
BOOTSTRAP_SEED = 20260906
FDR_Q = 0.05
SEEDS = [42, 137, 2718, 31415, 65537]
MODEL_KEYS = ["ebm", "tfidf_lr", "banglabert", "xlmr"]

# This is the actual Stage-05 execution order after CUE_FEATURE_SCHEMA.json
# was serialized with sort_keys=True and reloaded.
CUE_FAMILIES = [
    "curiosity_information_withholding",
    "direct_address_imperative",
    "exaggeration_superlative",
    "interrogative_structure",
    "punctuation_intensity",
    "sensational_emotional_loading",
    "specificity_numerals",
    "vague_deictic_reference",
]
PLATFORMS = ["news", "facebook", "youtube", "twitter"]

WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
    Path.cwd(),
]
OUT_ROOT = WORK_ROOT / "03_results" / "05_external_validation_H3_repair"
if OUT_ROOT.exists() and any(OUT_ROOT.iterdir()):
    raise RuntimeError(f"Refusing to overwrite existing H3 repair output: {OUT_ROOT}")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def cjb(o):
    return json.dumps(o, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")

def sha256_file(path, chunk=8*1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                return h.hexdigest()
            h.update(b)

def write_json(path, obj):
    Path(path).write_text(
        json.dumps(obj, ensure_ascii=False, sort_keys=True, indent=2) + "\n",
        encoding="utf-8",
        newline="\n",
    )

def write_text(path, text):
    Path(path).write_text(text, encoding="utf-8", newline="\n")

def hard_stop(message, details=None):
    write_json(
        OUT_ROOT / "H3_REPAIR_STATUS.json",
        {
            "stage_id": STAGE_ID,
            "repair_version": REPAIR_VERSION,
            "status": "HARD_STOP",
            "message": message,
            "details": details or {},
            "timestamp_utc": utc_now(),
        },
    )
    raise RuntimeError(message)

REQUIRED_SOURCE_FILES = [
    "LOCKED_SOURCE_TEST_PREDICTIONS.parquet",
    "BANGLACLICK_PLATFORM_PREDICTIONS.parquet",
    "BAITBUSTER_HUMAN_PREDICTIONS.parquet",
    "H3_RESULTS.json",
    "STATISTICAL_SUMMARY.json",
    "STAGE05_ARTIFACT_MANIFEST.json",
    "STAGE05_SHA256SUMS.txt",
    "STAGE_STATUS.json",
]

def parse_sumfile(path):
    out = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        h, rel = line.split(None, 1)
        rel = rel.lstrip("*").strip().replace("\\", "/")
        if not re.fullmatch(r"[0-9a-fA-F]{64}", h):
            hard_stop("Malformed Stage-05 checksum line", {"line": line})
        if rel in out:
            hard_stop("Duplicate path in Stage-05 checksum file", {"path": rel})
        out[rel] = h.lower()
    return out

def discover_stage05_root():
    candidates = set()
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for p in root.rglob("STAGE05_ARTIFACT_MANIFEST.json"):
            d = p.parent.resolve()
            if d == OUT_ROOT.resolve():
                continue
            if all((d / name).is_file() for name in REQUIRED_SOURCE_FILES):
                candidates.add(d)

    if not candidates:
        hard_stop(
            "Completed Stage-05 output root not found",
            {
                "required_files": REQUIRED_SOURCE_FILES,
                "searched_roots": [str(x) for x in SEARCH_ROOTS],
                "action": "Mount the already-successful Stage-05 output as a Kaggle input. Do not rerun model inference.",
            },
        )

    identities = defaultdict(list)
    for d in sorted(candidates, key=str):
        identity = {
            "manifest": sha256_file(d / "STAGE05_ARTIFACT_MANIFEST.json"),
            "source_predictions": sha256_file(d / "LOCKED_SOURCE_TEST_PREDICTIONS.parquet"),
            "banglaclick_predictions": sha256_file(d / "BANGLACLICK_PLATFORM_PREDICTIONS.parquet"),
            "baitbuster_predictions": sha256_file(d / "BAITBUSTER_HUMAN_PREDICTIONS.parquet"),
        }
        identities[hashlib.sha256(cjb(identity)).hexdigest()].append((d, identity))

    if len(identities) != 1:
        hard_stop(
            "Multiple distinct completed Stage-05 scientific identities found",
            {"candidate_roots": [str(x) for x in sorted(candidates, key=str)]},
        )

    group = next(iter(identities.values()))
    chosen, identity = sorted(group, key=lambda x: str(x[0]))[0]
    if len(group) > 1:
        print(f"Identical Stage-05 mounts collapsed: {len(group)}; using {chosen}", flush=True)
    return chosen, identity

SOURCE_ROOT, SOURCE_IDENTITY = discover_stage05_root()
print("H3 repair source root:", SOURCE_ROOT, flush=True)
print("Prediction identity:", json.dumps(SOURCE_IDENTITY, sort_keys=True), flush=True)


H3 repair source root: /kaggle/input/notebooks/smalakarishere/05-external-validation/03_results/05_external_validation
Prediction identity: {"baitbuster_predictions": "a3e95820c62de00984a189e2c250f7fffd223ad4b3954db04a5368c7a544b7b9", "banglaclick_predictions": "5685fd4142a0e8e7f85cd6040cf48042da2368d5f9596a7740c4f9f09c9519d5", "manifest": "2f0c503f0bb2a535172cc3f5e2f5a47cd42024db2ba865322276ff1c6fee6c00", "source_predictions": "df863d7a24f7d451451822890de58613c3295eb0bcc1aa882150137b0aa9880a"}


In [2]:
# ---------- Verify original Stage-05 provenance and immutable prediction artifacts ----------
source_status = json.loads((SOURCE_ROOT / "STAGE_STATUS.json").read_text(encoding="utf-8"))
if source_status.get("status") != "PASS":
    hard_stop("Source Stage-05 output is not PASS", {"observed_status": source_status})

source_manifest = json.loads((SOURCE_ROOT / "STAGE05_ARTIFACT_MANIFEST.json").read_text(encoding="utf-8"))
source_manifest_sha = sha256_file(SOURCE_ROOT / "STAGE05_ARTIFACT_MANIFEST.json")
source_sums = parse_sumfile(SOURCE_ROOT / "STAGE05_SHA256SUMS.txt")

# Verify the manifest payload seal if present.
recorded_payload = source_manifest.get("manifest_payload_sha256")
if recorded_payload:
    tmp = dict(source_manifest)
    tmp.pop("manifest_payload_sha256", None)
    observed_payload = hashlib.sha256(cjb(tmp)).hexdigest()
    if observed_payload != recorded_payload:
        hard_stop(
            "Source Stage-05 manifest payload hash mismatch",
            {"recorded": recorded_payload, "observed": observed_payload},
        )

PRED_FILES = [
    "LOCKED_SOURCE_TEST_PREDICTIONS.parquet",
    "BANGLACLICK_PLATFORM_PREDICTIONS.parquet",
    "BAITBUSTER_HUMAN_PREDICTIONS.parquet",
]
UNCHANGED_SCIENCE_FILES = [
    "H1_RESULTS.json",
    "H2_RESULTS.json",
    "H3_PERTURBATION_RESULTS.json",
    "SENSITIVITY_RESULTS.json",
]
for name in PRED_FILES + ["H3_RESULTS.json", "STATISTICAL_SUMMARY.json"] + UNCHANGED_SCIENCE_FILES:
    p = SOURCE_ROOT / name
    if not p.is_file():
        hard_stop("Required source Stage-05 artifact missing", {"artifact": name})
    observed = sha256_file(p)
    manifest_hash = source_manifest.get("artifacts", {}).get(name)
    checksum_hash = source_sums.get(name)
    if manifest_hash is not None and observed != manifest_hash:
        hard_stop("Source artifact does not match Stage-05 manifest", {"artifact": name})
    if checksum_hash is not None and observed != checksum_hash:
        hard_stop("Source artifact does not match Stage-05 checksum file", {"artifact": name})

SRC_P = pd.read_parquet(SOURCE_ROOT / PRED_FILES[0])
BC_P = pd.read_parquet(SOURCE_ROOT / PRED_FILES[1])
BB_P = pd.read_parquet(SOURCE_ROOT / PRED_FILES[2])

required_common = {
    "row_id","title","platform","label","cluster_id","dataset","model","seed",
    "logit","p_uncalibrated","temperature","p_calibrated","normalized_title_sha256",
    *CUE_FAMILIES,
}
for name, df in [("source", SRC_P), ("banglaclick", BC_P), ("baitbuster", BB_P)]:
    missing = sorted(required_common - set(df.columns))
    if missing:
        hard_stop("Prediction artifact schema incomplete", {"domain": name, "missing": missing})
    if df["row_id"].isna().any() or df["cluster_id"].isna().any():
        hard_stop("Prediction artifact has missing row/cluster identity", {"domain": name})
    if set(df["model"].astype(str).unique()) != set(MODEL_KEYS):
        hard_stop("Prediction artifact model panel mismatch", {"domain": name})
    if set(df["seed"].astype(int).unique()) != set(SEEDS):
        hard_stop("Prediction artifact seed panel mismatch", {"domain": name})
    if not set(df["label"].astype(int).unique()).issubset({0,1}):
        hard_stop("Prediction artifact label is not binary", {"domain": name})
    if not np.isfinite(df["p_calibrated"].to_numpy(float)).all():
        hard_stop("Prediction artifact has non-finite calibrated probabilities", {"domain": name})
    if ((df["p_calibrated"].to_numpy(float) <= 0) | (df["p_calibrated"].to_numpy(float) >= 1)).any():
        hard_stop("Prediction artifact calibrated probabilities are outside open interval (0,1)", {"domain": name})

expected_item_rows = {"source":2784, "banglaclick":4961, "baitbuster":10000}
for name, df in [("source", SRC_P), ("banglaclick", BC_P), ("baitbuster", BB_P)]:
    for (model, seed), g in df.groupby(["model","seed"], sort=True):
        if len(g) != expected_item_rows[name] or g["row_id"].nunique() != expected_item_rows[name]:
            hard_stop(
                "Prediction artifact model/seed row contract mismatch",
                {"domain":name,"model":str(model),"seed":int(seed),"rows":int(len(g)),
                 "unique_row_ids":int(g["row_id"].nunique()),"expected":expected_item_rows[name]},
            )

# Static fields must not change across model/seed replicas.
static_cols = ["title","platform","label","cluster_id","normalized_title_sha256",*CUE_FAMILIES]
def validate_static_replication(df, domain):
    canonical = (
        df[(df["model"]=="ebm") & (df["seed"].astype(int)==SEEDS[0])]
        .sort_values("row_id", kind="stable")
        .set_index("row_id")[static_cols]
    )
    for (model, seed), g in df.groupby(["model","seed"], sort=True):
        x = g.sort_values("row_id", kind="stable").set_index("row_id")[static_cols]
        if not x.index.equals(canonical.index):
            hard_stop("Prediction row identities differ across model/seed replicas",
                      {"domain":domain,"model":str(model),"seed":int(seed)})
        if not x.equals(canonical):
            hard_stop("Labels/clusters/cues differ across model/seed replicas",
                      {"domain":domain,"model":str(model),"seed":int(seed)})
    return canonical.reset_index()

src_static = validate_static_replication(SRC_P, "source")
bc_static = validate_static_replication(BC_P, "banglaclick")
bb_static = validate_static_replication(BB_P, "baitbuster")

if set(bc_static["platform"].unique()) != set(PLATFORMS):
    hard_stop("BanglaClick platform set mismatch", {"observed":sorted(map(str,bc_static["platform"].unique()))})
if not src_static["cluster_id"].astype(str).str.startswith("source_row:").all():
    hard_stop("Source TEST clustering is not the locked row-level fallback")
if not bc_static["cluster_id"].astype(str).str.startswith("bc:").all():
    hard_stop("BanglaClick clustering identity is not the locked Stage-05 form")
if not bb_static["cluster_id"].astype(str).str.startswith("bb:").all():
    hard_stop("BaitBuster clustering is not channel_id-based Stage-05 form")

# Confirm the original H3 contrast order, which fixes RNG consumption across contrasts.
old_h3 = json.loads((SOURCE_ROOT / "H3_RESULTS.json").read_text(encoding="utf-8"))
old_rows = old_h3.get("banglaclick_cue_platform_contrasts", [])
old_order = [(str(r["cue"]), str(r["target"]).split(":",1)[1]) for r in old_rows]
expected_order = [(cue,p) for cue in CUE_FAMILIES for p in PLATFORMS]
if old_order != expected_order:
    hard_stop(
        "Original successful H3 contrast order differs from the frozen repair order",
        {"observed_first":old_order[:12],"expected_first":expected_order[:12]},
    )

print("Immutable Stage-05 prediction/provenance validation: PASS", flush=True)


Immutable Stage-05 prediction/provenance validation: PASS


In [3]:
# ---------- Strict H3 statistical recomputation; no model inference ----------
class ClusterBootstrapPlan:
    def __init__(self, cluster_ids, name):
        clusters = np.asarray(cluster_ids).astype(str)
        if len(clusters) == 0:
            hard_stop("Cannot construct bootstrap plan for empty domain", {"domain":name})
        uniq = np.array(sorted(set(clusters)))
        code = {u:i for i,u in enumerate(uniq)}
        self.row_cluster = np.asarray([code[u] for u in clusters], dtype=np.int32)
        self.cluster_labels = uniq
        self.n_clusters = int(len(uniq))
        self.n_rows = int(len(clusters))
        self.name = str(name)

    def draw_row_counts(self, rng):
        sampled_codes = rng.choice(self.n_clusters, size=self.n_clusters, replace=True)
        multiplicity = np.bincount(sampled_codes, minlength=self.n_clusters)
        return multiplicity[self.row_cluster]

def item_frame(pred):
    return (
        pred[(pred["model"]=="ebm") & (pred["seed"].astype(int)==SEEDS[0])]
        .drop_duplicates("row_id")
        .sort_values("row_id", kind="stable")
        .reset_index(drop=True)
    )

def ebm_seed_frames(pred):
    frames = {}
    canonical = None
    for seed in SEEDS:
        g = (
            pred[(pred["model"]=="ebm") & (pred["seed"].astype(int)==seed)]
            .sort_values("row_id", kind="stable")
            .reset_index(drop=True)
        )
        ids = g["row_id"].astype(str).tolist()
        if canonical is None:
            canonical = ids
        elif ids != canonical:
            hard_stop("EBM seed predictions are not row-aligned", {"seed":seed})
        frames[seed] = g
    return frames

def h3_bundle(item, seed_frames, name):
    ids = item["row_id"].astype(str).tolist()
    y = item["label"].to_numpy(int)
    cues = {cue:(item[cue].to_numpy() > 0) for cue in CUE_FAMILIES}
    losses = []
    for seed in SEEDS:
        g = seed_frames[seed]
        if g["row_id"].astype(str).tolist() != ids:
            hard_stop("H3 EBM seed frame row misalignment", {"domain":name,"seed":seed})
        if not np.array_equal(g["label"].to_numpy(int), y):
            hard_stop("H3 labels differ across EBM seeds", {"domain":name,"seed":seed})
        p = np.clip(g["p_calibrated"].to_numpy(float), 1e-7, 1-1e-7)
        loss = -(y*np.log(p) + (1-y)*np.log(1-p))
        if not np.isfinite(loss).all():
            hard_stop("Non-finite H3 proper loss", {"domain":name,"seed":seed})
        losses.append(loss)
    return {
        "name":name,
        "row_ids":ids,
        "y":y,
        "cues":cues,
        "loss_by_seed":np.vstack(losses),
        "plan":ClusterBootstrapPlan(item["cluster_id"].to_numpy(), name),
        "ones":np.ones(len(item), dtype=np.int16),
    }

def support_record(bundle, cue):
    mask = bundle["cues"][cue]
    clusters = np.asarray(bundle["plan"].cluster_labels)
    row_cluster = bundle["plan"].row_cluster
    present_codes = set(row_cluster[mask].tolist())
    absent_codes = set(row_cluster[~mask].tolist())
    K = bundle["plan"].n_clusters
    k_present = len(present_codes)
    k_absent = len(absent_codes)
    p_no_present = ((K-k_present)/K)**K if K else None
    p_no_absent = ((K-k_absent)/K)**K if K else None
    return {
        "rows":int(bundle["plan"].n_rows),
        "clusters":int(K),
        "cue_present_rows":int(mask.sum()),
        "cue_absent_rows":int((~mask).sum()),
        "clusters_with_cue_present":int(k_present),
        "clusters_with_cue_absent":int(k_absent),
        "analytical_p_resample_missing_cue_present":float(p_no_present),
        "analytical_p_resample_missing_cue_absent":float(p_no_absent),
    }

def group_support_reasons(bundle, cue, row_counts, prefix):
    mask = bundle["cues"][cue]
    reasons = []
    if float(np.asarray(row_counts)[mask].sum()) <= 0:
        reasons.append(f"{prefix}_missing_cue_present")
    if float(np.asarray(row_counts)[~mask].sum()) <= 0:
        reasons.append(f"{prefix}_missing_cue_absent")
    return reasons

def group_gap_1d(values, cue_present, row_counts):
    values = np.asarray(values, float)
    cue_present = np.asarray(cue_present, bool)
    w = np.asarray(row_counts, float)
    wp = w[cue_present]
    wa = w[~cue_present]
    dp = float(wp.sum())
    da = float(wa.sum())
    if dp <= 0 or da <= 0:
        raise ValueError("group_gap_1d called on a non-estimable resample")
    return float(np.dot(values[cue_present], wp)/dp - np.dot(values[~cue_present], wa)/da)

def group_gap_by_seed(loss_by_seed, cue_present, row_counts):
    loss = np.asarray(loss_by_seed, float)
    cue_present = np.asarray(cue_present, bool)
    w = np.asarray(row_counts, float)
    wp = w[cue_present]
    wa = w[~cue_present]
    dp = float(wp.sum())
    da = float(wa.sum())
    if dp <= 0 or da <= 0:
        raise ValueError("group_gap_by_seed called on a non-estimable resample")
    return (loss[:,cue_present] @ wp)/dp - (loss[:,~cue_present] @ wa)/da

def complete_ci(values):
    a = np.asarray(values, float)
    if len(a) != BOOTSTRAP_REPS or not np.isfinite(a).all():
        raise ValueError("Confirmatory CI requires all 5,000 finite locked bootstrap replicates")
    return [float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))]

def complete_p(values):
    a = np.asarray(values, float)
    if len(a) != BOOTSTRAP_REPS or not np.isfinite(a).all():
        raise ValueError("Confirmatory bootstrap p-value requires all 5,000 finite locked replicates")
    return float(min(1.0, 2*min(np.mean(a <= 0), np.mean(a >= 0))))

def simes_complete(pvals):
    if len(pvals) != 4 or not all(p is not None and np.isfinite(p) for p in pvals):
        return None
    p = np.sort(np.asarray(pvals, float))
    return float(min(1.0, np.min(p*len(p)/np.arange(1, len(p)+1))))

def invalidity_reason_text():
    return (
        "At least one ordinary locked cluster-bootstrap replicate contained zero total "
        "multiplicity in the cue-present or cue-absent group in source and/or target. "
        "The difference-of-group-means statistic is undefined for that replicate. "
        "The locked protocol did not prespecify rejection resampling or conditioning on "
        "valid draws, so confirmatory CI/p are not computed from a post-hoc subset."
    )

src_item = item_frame(SRC_P)
bc_items = {p:item_frame(BC_P[BC_P["platform"].eq(p)]) for p in PLATFORMS}
bb_item = item_frame(BB_P)
src_ebm = ebm_seed_frames(SRC_P)
bc_ebm = {p:ebm_seed_frames(BC_P[BC_P["platform"].eq(p)]) for p in PLATFORMS}
bb_ebm = ebm_seed_frames(BB_P)

src_h3 = h3_bundle(src_item, src_ebm, "source_test")
bc_h3 = {p:h3_bundle(bc_items[p], bc_ebm[p], f"banglaclick:{p}") for p in PLATFORMS}
bb_h3 = h3_bundle(bb_item, bb_ebm, "baitbuster:youtube_human")

# Observed support is written before bootstrap so sparsity is auditable independently.
sparsity_rows = []
for cue in CUE_FAMILIES:
    sparsity_rows.append({"domain":"source_test","cue":cue,**support_record(src_h3,cue)})
    for p in PLATFORMS:
        sparsity_rows.append({"domain":f"banglaclick:{p}","cue":cue,**support_record(bc_h3[p],cue)})
    sparsity_rows.append({"domain":"baitbuster:youtube_human","cue":cue,**support_record(bb_h3,cue)})
pd.DataFrame(sparsity_rows).to_csv(OUT_ROOT/"H3_SPARSITY_REPORT.csv", index=False)

# ---- 32 prespecified BanglaClick cue/platform contrasts ----
rng_h3 = np.random.default_rng(BOOTSTRAP_SEED + 1)
cue_transport = []
print("[H3 repair] BanglaClick cue/platform bootstrap START", flush=True)

for ci, (cue, p) in enumerate([(c,p) for c in CUE_FAMILIES for p in PLATFORMS], 1):
    tgt = bc_h3[p]

    # Point estimate must itself be defined.
    point_reasons = (
        group_support_reasons(src_h3, cue, src_h3["ones"], "source")
        + group_support_reasons(tgt, cue, tgt["ones"], "target")
    )
    if point_reasons:
        point_rd = None
        point_nll = None
        by_seed_json = [None]*len(SEEDS)
    else:
        src_rd = group_gap_1d(src_h3["y"], src_h3["cues"][cue], src_h3["ones"])
        tgt_rd = group_gap_1d(tgt["y"], tgt["cues"][cue], tgt["ones"])
        point_rd = float(tgt_rd - src_rd)
        src_nll_seed = group_gap_by_seed(src_h3["loss_by_seed"], src_h3["cues"][cue], src_h3["ones"])
        tgt_nll_seed = group_gap_by_seed(tgt["loss_by_seed"], tgt["cues"][cue], tgt["ones"])
        by_seed = tgt_nll_seed - src_nll_seed
        point_nll = float(np.mean(by_seed))
        by_seed_json = [float(x) for x in by_seed]

    rd_boot = []
    nll_boot = []
    invalid_reason_counts = defaultdict(int)

    for rep in range(1, BOOTSTRAP_REPS+1):
        sw = src_h3["plan"].draw_row_counts(rng_h3)
        tw = tgt["plan"].draw_row_counts(rng_h3)

        reasons = (
            group_support_reasons(src_h3, cue, sw, "source")
            + group_support_reasons(tgt, cue, tw, "target")
        )
        if reasons:
            invalid_reason_counts["+".join(sorted(reasons))] += 1
        else:
            rd_boot.append(
                group_gap_1d(tgt["y"], tgt["cues"][cue], tw)
                - group_gap_1d(src_h3["y"], src_h3["cues"][cue], sw)
            )
            sd = group_gap_by_seed(src_h3["loss_by_seed"], src_h3["cues"][cue], sw)
            td = group_gap_by_seed(tgt["loss_by_seed"], tgt["cues"][cue], tw)
            nll_boot.append(float(np.mean(td-sd)))

        if rep % 1000 == 0:
            print(f"[H3 repair] contrast {ci}/32 {cue} {p}: {rep}/5000", flush=True)

    invalid = int(sum(invalid_reason_counts.values()))
    valid = BOOTSTRAP_REPS - invalid
    estimable = (not point_reasons) and invalid == 0
    status = "ESTIMABLE" if estimable else "NON_ESTIMABLE_UNDER_LOCKED_BOOTSTRAP"

    if estimable:
        rd_ci = complete_ci(rd_boot)
        rd_p = complete_p(rd_boot)
        nll_ci = complete_ci(nll_boot)
        nll_p = complete_p(nll_boot)
        reason = None
    else:
        rd_ci = rd_p = nll_ci = nll_p = None
        reason = (
            "Observed point statistic is undefined: " + ", ".join(point_reasons)
            if point_reasons else invalidity_reason_text()
        )

    cue_transport.append({
        "cue":cue,
        "target":f"banglaclick:{p}",
        "source_rd":None if point_reasons else float(src_rd),
        "target_rd":None if point_reasons else float(tgt_rd),
        "delta_rd":point_rd,
        "delta_rd_ci95":rd_ci,
        "delta_rd_p":rd_p,
        "ebm_delta_nll_gap_mean_5seeds":point_nll,
        "ebm_delta_nll_gap_ci95":nll_ci,
        "ebm_delta_nll_gap_p":nll_p,
        "ebm_delta_nll_gap_by_seed":by_seed_json,
        "bootstrap_attempts":BOOTSTRAP_REPS,
        "valid_replicates":int(valid),
        "invalid_replicates":int(invalid),
        "invalid_rate":float(invalid/BOOTSTRAP_REPS),
        "invalid_reason_counts":dict(sorted(invalid_reason_counts.items())),
        "inferential_status":status,
        "reason_for_nonestimability":reason,
        "source_support":support_record(src_h3,cue),
        "target_support":support_record(tgt,cue),
    })

# ---- Family Simes + BH-FDR without post-hoc family deletion ----
def family_simes_rows(endpoint_p_key):
    out = {}
    for cue in CUE_FAMILIES:
        rows = [r for r in cue_transport if r["cue"] == cue]
        pvals = [r[endpoint_p_key] for r in rows]
        p = simes_complete(pvals)
        out[cue] = {
            "status":"ESTIMABLE" if p is not None else "NON_ESTIMABLE_INCOMPLETE_FOUR_PLATFORM_FAMILY",
            "simes_p":p,
            "component_p_values":pvals,
            "nonestimable_platforms":[r["target"] for r in rows if r["inferential_status"]!="ESTIMABLE"],
        }
    return out

family_outcome = family_simes_rows("delta_rd_p")
family_reliability = family_simes_rows("ebm_delta_nll_gap_p")

def full_bh(family_rows):
    pvals = [family_rows[c]["simes_p"] for c in CUE_FAMILIES]
    if not all(p is not None and np.isfinite(p) for p in pvals):
        rows = {}
        for cue in CUE_FAMILIES:
            rows[cue] = {
                "simes_p":family_rows[cue]["simes_p"],
                "bh_q":None,
                "reject_q_0_05":None,
                "status":"BH_NOT_RUN_FULL_PRESPECIFIED_FAMILY_INCOMPLETE",
            }
        return {
            "status":"NON_ESTIMABLE_FULL_8_FAMILY_BH",
            "q":FDR_Q,
            "number_prespecified_families":8,
            "number_estimable_family_p_values":int(sum(p is not None for p in pvals)),
            "reason":"BH-FDR was not run on a post-hoc subset; at least one prespecified cue-family Simes p-value is unavailable.",
            "rows":rows,
        }
    reject, qvals, _, _ = multipletests(pvals, alpha=FDR_Q, method="fdr_bh")
    return {
        "status":"ESTIMABLE",
        "q":FDR_Q,
        "number_prespecified_families":8,
        "number_estimable_family_p_values":8,
        "rows":{
            cue:{
                "simes_p":float(p),
                "bh_q":float(q),
                "reject_q_0_05":bool(r),
                "status":"ESTIMABLE",
            }
            for cue,p,q,r in zip(CUE_FAMILIES,pvals,qvals,reject)
        },
    }

family_fdr_outcome_full = full_bh(family_outcome)
family_fdr_reliability_full = full_bh(family_reliability)

# Compatibility dictionaries retain the historical cue-key surface but never
# pretend q-values exist when the full eight-family procedure is non-estimable.
family_fdr_outcome = family_fdr_outcome_full["rows"]
family_fdr_reliability = family_fdr_reliability_full["rows"]

# ---- Full prespecified 32-dimensional BanglaClick outcome Wald test ----
point_vec = np.asarray([
    next(r["delta_rd"] for r in cue_transport if r["cue"]==cue and r["target"]==f"banglaclick:{p}")
    for cue in CUE_FAMILIES for p in PLATFORMS
], dtype=float)

B = np.full((BOOTSTRAP_REPS, 32), np.nan, dtype=float)
global_reason_counts = [defaultdict(int) for _ in range(32)]
rng_global = np.random.default_rng(BOOTSTRAP_SEED + 2)

print("[H3 repair] full 32-D global Wald bootstrap START", flush=True)
for rep in range(BOOTSTRAP_REPS):
    sw = src_h3["plan"].draw_row_counts(rng_global)
    tw = {p:bc_h3[p]["plan"].draw_row_counts(rng_global) for p in PLATFORMS}
    j = 0
    for cue in CUE_FAMILIES:
        source_reasons = group_support_reasons(src_h3, cue, sw, "source")
        for p in PLATFORMS:
            tgt = bc_h3[p]
            reasons = source_reasons + group_support_reasons(tgt, cue, tw[p], "target")
            if reasons:
                global_reason_counts[j]["+".join(sorted(reasons))] += 1
            else:
                B[rep,j] = (
                    group_gap_1d(tgt["y"], tgt["cues"][cue], tw[p])
                    - group_gap_1d(src_h3["y"], src_h3["cues"][cue], sw)
                )
            j += 1
    if (rep+1) % 1000 == 0:
        print(f"[H3 repair] global Wald: {rep+1}/5000", flush=True)

dimension_records = []
j = 0
for cue in CUE_FAMILIES:
    for p in PLATFORMS:
        invalid_count = int((~np.isfinite(B[:,j])).sum())
        dimension_records.append({
            "dimension_index":j,
            "cue":cue,
            "platform":p,
            "invalid_replicates":invalid_count,
            "valid_replicates":int(BOOTSTRAP_REPS-invalid_count),
            "invalid_rate":float(invalid_count/BOOTSTRAP_REPS),
            "invalid_reason_counts":dict(sorted(global_reason_counts[j].items())),
        })
        j += 1

joint_valid = np.isfinite(B).all(axis=1)
point_finite = np.isfinite(point_vec).all()
all_boot_finite = bool(joint_valid.all())

global_result = {
    "status":None,
    "expected_dimension":32,
    "dimension_reduction_performed":False,
    "bootstrap_attempts":BOOTSTRAP_REPS,
    "joint_valid_replicates":int(joint_valid.sum()),
    "joint_invalid_replicates":int((~joint_valid).sum()),
    "dimension_estimability":dimension_records,
    "statistic":None,
    "df":None,
    "p":None,
    "covariance_rank":None,
    "reason":None,
}

if not point_finite:
    global_result["status"] = "NON_ESTIMABLE"
    global_result["reason"] = "At least one prespecified 32-D observed point component is undefined."
elif not all_boot_finite:
    global_result["status"] = "NON_ESTIMABLE"
    global_result["reason"] = (
        "At least one of the 5,000 locked joint cluster-bootstrap vectors contains an undefined "
        "prespecified dimension. No dimensions were dropped and no reduced post-hoc Wald test was run."
    )
else:
    cov = np.atleast_2d(np.cov(B, rowvar=False))
    rank = int(np.linalg.matrix_rank(cov))
    global_result["covariance_rank"] = rank
    if rank != 32:
        global_result["status"] = "NON_ESTIMABLE"
        global_result["reason"] = (
            f"The complete finite 32×32 bootstrap covariance has rank {rank}, not 32. "
            "No rank-reduced post-hoc Wald test was substituted."
        )
    else:
        try:
            wald = float(point_vec @ np.linalg.solve(cov, point_vec))
        except np.linalg.LinAlgError:
            global_result["status"] = "NON_ESTIMABLE"
            global_result["reason"] = "The full 32×32 covariance could not be stably solved."
        else:
            global_result.update({
                "status":"ESTIMABLE",
                "statistic":wald,
                "df":32,
                "p":float(chi2.sf(wald, 32)),
                "reason":None,
            })

# ---- Independent BaitBuster replication; same strict estimability reporting ----
bb_rep = []
rng_rep = np.random.default_rng(BOOTSTRAP_SEED + 3)
print("[H3 repair] BaitBuster replication bootstrap START", flush=True)

for cue_i, cue in enumerate(CUE_FAMILIES, 1):
    point_reasons = (
        group_support_reasons(src_h3, cue, src_h3["ones"], "source")
        + group_support_reasons(bb_h3, cue, bb_h3["ones"], "target")
    )
    if point_reasons:
        point_rd = point_nll = None
        by_seed_json = [None]*len(SEEDS)
    else:
        src_rd = group_gap_1d(src_h3["y"], src_h3["cues"][cue], src_h3["ones"])
        tgt_rd = group_gap_1d(bb_h3["y"], bb_h3["cues"][cue], bb_h3["ones"])
        point_rd = float(tgt_rd-src_rd)
        src_nll = group_gap_by_seed(src_h3["loss_by_seed"], src_h3["cues"][cue], src_h3["ones"])
        tgt_nll = group_gap_by_seed(bb_h3["loss_by_seed"], bb_h3["cues"][cue], bb_h3["ones"])
        by_seed = tgt_nll-src_nll
        point_nll = float(np.mean(by_seed))
        by_seed_json = [float(x) for x in by_seed]

    rd_boot = []
    nll_boot = []
    reason_counts = defaultdict(int)

    for rep in range(1, BOOTSTRAP_REPS+1):
        sw = src_h3["plan"].draw_row_counts(rng_rep)
        tw = bb_h3["plan"].draw_row_counts(rng_rep)
        reasons = (
            group_support_reasons(src_h3, cue, sw, "source")
            + group_support_reasons(bb_h3, cue, tw, "target")
        )
        if reasons:
            reason_counts["+".join(sorted(reasons))] += 1
        else:
            rd_boot.append(
                group_gap_1d(bb_h3["y"], bb_h3["cues"][cue], tw)
                - group_gap_1d(src_h3["y"], src_h3["cues"][cue], sw)
            )
            sd = group_gap_by_seed(src_h3["loss_by_seed"], src_h3["cues"][cue], sw)
            td = group_gap_by_seed(bb_h3["loss_by_seed"], bb_h3["cues"][cue], tw)
            nll_boot.append(float(np.mean(td-sd)))
        if rep % 1000 == 0:
            print(f"[H3 repair] BaitBuster {cue_i}/8 {cue}: {rep}/5000", flush=True)

    invalid = int(sum(reason_counts.values()))
    valid = BOOTSTRAP_REPS-invalid
    estimable = (not point_reasons) and invalid == 0
    bb_rep.append({
        "cue":cue,
        "delta_rd":point_rd,
        "delta_rd_ci95":complete_ci(rd_boot) if estimable else None,
        "delta_rd_p":complete_p(rd_boot) if estimable else None,
        "ebm_delta_nll_gap_mean_5seeds":point_nll,
        "ebm_delta_nll_gap_ci95":complete_ci(nll_boot) if estimable else None,
        "ebm_delta_nll_gap_p":complete_p(nll_boot) if estimable else None,
        "ebm_delta_nll_gap_by_seed":by_seed_json,
        "bootstrap_attempts":BOOTSTRAP_REPS,
        "valid_replicates":int(valid),
        "invalid_replicates":int(invalid),
        "invalid_rate":float(invalid/BOOTSTRAP_REPS),
        "invalid_reason_counts":dict(sorted(reason_counts.items())),
        "inferential_status":"ESTIMABLE" if estimable else "NON_ESTIMABLE_UNDER_LOCKED_BOOTSTRAP",
        "reason_for_nonestimability":None if estimable else (
            "Observed point statistic is undefined: " + ", ".join(point_reasons)
            if point_reasons else invalidity_reason_text()
        ),
        "source_support":support_record(src_h3,cue),
        "target_support":support_record(bb_h3,cue),
    })

nonestimable_bc = [r for r in cue_transport if r["inferential_status"]!="ESTIMABLE"]
nonestimable_bb = [r for r in bb_rep if r["inferential_status"]!="ESTIMABLE"]

H3_CORRECTED = {
    "schema_version":3,
    "repair_version":REPAIR_VERSION,
    "repair_scope":"H3 statistical layer only; immutable Stage-05 predictions consumed read-only",
    "source_stage05_manifest_sha256":source_manifest_sha,
    "prediction_artifact_sha256":SOURCE_IDENTITY,
    "bootstrap_replicates":BOOTSTRAP_REPS,
    "bootstrap_seed":BOOTSTRAP_SEED,
    "ci":"95% percentile; confirmatory only when all 5,000 locked replicates are finite",
    "cluster_rule":"unchanged from Stage 05: strongest available source entity; source TEST=row; BanglaClick=stored SOURCE_URL-derived cluster_id; BaitBuster=stored channel_id-derived cluster_id",
    "primary":"prespecified cue–outcome and cue-conditioned proper-loss transport",
    "proper_loss":"NLL on frozen temperature-scaled deployment probabilities",
    "estimability_policy":{
        "conditional_valid_only_bootstrap_used":False,
        "rejection_resampling_used":False,
        "posthoc_dimension_reduction_used":False,
        "contrast_rule":"Any invalid locked replicate => confirmatory CI/p NON_ESTIMABLE.",
        "global_rule":"All 32 dimensions and all 5,000 joint replicates plus full-rank 32×32 covariance are required.",
        "bh_rule":"BH-FDR is not run on a post-hoc subset of the eight prespecified cue families.",
    },
    "global_banglaclick_outcome_wald":global_result,
    "global_banglaclick_wald":global_result,
    "banglaclick_cue_platform_contrasts":cue_transport,
    "family_simes_outcome":family_outcome,
    "family_simes_reliability":family_reliability,
    "family_fdr_outcome_full":family_fdr_outcome_full,
    "family_fdr_reliability_full":family_fdr_reliability_full,
    "family_fdr_outcome":family_fdr_outcome,
    "family_fdr_reliability":family_fdr_reliability,
    "family_fdr":family_fdr_outcome,
    "baitbuster_replication":bb_rep,
    "nonestimable_banglaclick_contrast_count":len(nonestimable_bc),
    "nonestimable_baitbuster_replication_count":len(nonestimable_bb),
    "causal_language":False,
    "interpretation":"association/transport of predictive outcome and proper-loss relationships under observed distribution shift",
}
write_json(OUT_ROOT/"H3_RESULTS.json", H3_CORRECTED)
pd.DataFrame(cue_transport).to_csv(OUT_ROOT/"H3_CUE_PLATFORM_CONTRASTS_CORRECTED.csv", index=False)
pd.DataFrame(bb_rep).to_csv(OUT_ROOT/"H3_BAITBUSTER_REPLICATION_CORRECTED.csv", index=False)

print(
    "[H3 repair] COMPLETE — "
    f"BanglaClick non-estimable contrasts={len(nonestimable_bc)}/32; "
    f"global={global_result['status']}; "
    f"BaitBuster non-estimable cues={len(nonestimable_bb)}/8",
    flush=True,
)


[H3 repair] BanglaClick cue/platform bootstrap START
[H3 repair] contrast 1/32 curiosity_information_withholding news: 1000/5000
[H3 repair] contrast 1/32 curiosity_information_withholding news: 2000/5000
[H3 repair] contrast 1/32 curiosity_information_withholding news: 3000/5000
[H3 repair] contrast 1/32 curiosity_information_withholding news: 4000/5000
[H3 repair] contrast 1/32 curiosity_information_withholding news: 5000/5000
[H3 repair] contrast 2/32 curiosity_information_withholding facebook: 1000/5000
[H3 repair] contrast 2/32 curiosity_information_withholding facebook: 2000/5000
[H3 repair] contrast 2/32 curiosity_information_withholding facebook: 3000/5000
[H3 repair] contrast 2/32 curiosity_information_withholding facebook: 4000/5000
[H3 repair] contrast 2/32 curiosity_information_withholding facebook: 5000/5000
[H3 repair] contrast 3/32 curiosity_information_withholding youtube: 1000/5000
[H3 repair] contrast 3/32 curiosity_information_withholding youtube: 2000/5000
[H3 repai

In [4]:
# ---------- Updated statistical summary, provenance manifest, and repair decision ----------
source_summary_path = SOURCE_ROOT / "STATISTICAL_SUMMARY.json"
source_summary = json.loads(source_summary_path.read_text(encoding="utf-8"))
corrected_summary = copy.deepcopy(source_summary)
corrected_summary["h3_repair"] = {
    "repair_version":REPAIR_VERSION,
    "source_statistical_summary_sha256":sha256_file(source_summary_path),
    "corrected_h3_results_sha256":sha256_file(OUT_ROOT/"H3_RESULTS.json"),
    "only_h3_statistics_recomputed":True,
    "model_inference_rerun":False,
    "calibration_refit":False,
    "h1_recomputed":False,
    "h2_recomputed":False,
    "h3p_recomputed":False,
    "h4_recomputed":False,
    "banglaclick_nonestimable_contrasts":H3_CORRECTED["nonestimable_banglaclick_contrast_count"],
    "baitbuster_nonestimable_replications":H3_CORRECTED["nonestimable_baitbuster_replication_count"],
    "global_wald_status":global_result["status"],
    "full_32d_estimable":bool(global_result["status"]=="ESTIMABLE"),
    "family_bh_outcome_status":family_fdr_outcome_full["status"],
    "family_bh_reliability_status":family_fdr_reliability_full["status"],
}
write_json(OUT_ROOT/"STATISTICAL_SUMMARY.json", corrected_summary)

source_artifact_hashes = {
    name:sha256_file(SOURCE_ROOT/name)
    for name in PRED_FILES + ["H3_RESULTS.json","STATISTICAL_SUMMARY.json"] + UNCHANGED_SCIENCE_FILES
}
recomputed = [
    "H3_RESULTS.json",
    "H3_SPARSITY_REPORT.csv",
    "H3_CUE_PLATFORM_CONTRASTS_CORRECTED.csv",
    "H3_BAITBUSTER_REPLICATION_CORRECTED.csv",
    "STATISTICAL_SUMMARY.json",
]
recomputed_hashes = {name:sha256_file(OUT_ROOT/name) for name in recomputed}

manifest = {
    "schema_version":1,
    "stage_id":STAGE_ID,
    "repair_version":REPAIR_VERSION,
    "timestamp_utc":utc_now(),
    "source_stage05_root":str(SOURCE_ROOT),
    "source_stage05_manifest_sha256":source_manifest_sha,
    "source_stage05_status_sha256":sha256_file(SOURCE_ROOT/"STAGE_STATUS.json"),
    "source_artifacts_read_only":source_artifact_hashes,
    "recomputed_artifacts":recomputed_hashes,
    "boundary":{
        "only_h3_statistical_layer_recomputed":True,
        "stage03_rerun":False,
        "stage04_rerun":False,
        "model_inference_rerun":False,
        "prediction_files_modified":False,
        "h1_modified":False,
        "h2_modified":False,
        "h3p_modified":False,
        "h4_modified":False,
    },
    "constants":{
        "bootstrap_reps":BOOTSTRAP_REPS,
        "bootstrap_seed":BOOTSTRAP_SEED,
        "seeds":SEEDS,
        "cue_families":CUE_FAMILIES,
        "platforms":PLATFORMS,
        "fdr_q":FDR_Q,
    },
    "decision":{
        "h3_repair_execution":"PASS",
        "full_h3_32d_estimable":bool(global_result["status"]=="ESTIMABLE"),
        "global_wald_status":global_result["status"],
        "nonestimable_banglaclick_contrasts":len(nonestimable_bc),
        "paper_interpretation_boundary":(
            "Full H3 global inference may be interpreted only if global_wald_status == ESTIMABLE. "
            "Non-estimable contrasts must be reported as sparsity/estimability limitations; "
            "their old conditional-valid CIs/p-values are superseded."
        ),
    },
}
manifest["manifest_payload_sha256"] = hashlib.sha256(cjb(manifest)).hexdigest()
write_json(OUT_ROOT/"STAGE05_H3_REPAIR_MANIFEST.json", manifest)

status = {
    "stage_id":STAGE_ID,
    "repair_version":REPAIR_VERSION,
    "status":"PASS",
    "full_h3_estimable":bool(global_result["status"]=="ESTIMABLE"),
    "global_wald_status":global_result["status"],
    "number_nonestimable_banglaclick_cue_platform_contrasts":len(nonestimable_bc),
    "number_nonestimable_baitbuster_replications":len(nonestimable_bb),
    "corrected_h3_results":str(OUT_ROOT/"H3_RESULTS.json"),
    "timestamp_utc":utc_now(),
}
write_json(OUT_ROOT/"H3_REPAIR_STATUS.json", status)

# Final checksum file covers only repair outputs; source Stage-05 remains untouched.
files = sorted(
    p for p in OUT_ROOT.iterdir()
    if p.is_file() and p.name != "H3_REPAIR_SHA256SUMS.txt"
)
write_text(
    OUT_ROOT/"H3_REPAIR_SHA256SUMS.txt",
    "\n".join(f"{sha256_file(p)}  {p.name}" for p in files) + "\n",
)

print(json.dumps(status, indent=2), flush=True)
print("H3-only repair output:", OUT_ROOT, flush=True)


{
  "stage_id": "05-H3-REPAIR",
  "repair_version": "SWARABYANJAN-H3-ESTIMABILITY-REPAIR-v1.0.0",
  "status": "PASS",
  "full_h3_estimable": false,
  "global_wald_status": "NON_ESTIMABLE",
  "number_nonestimable_banglaclick_cue_platform_contrasts": 8,
  "number_nonestimable_baitbuster_replications": 3,
  "corrected_h3_results": "/kaggle/working/03_results/05_external_validation_H3_repair/H3_RESULTS.json",
  "timestamp_utc": "2026-09-11T08:59:50.450657+00:00"
}
H3-only repair output: /kaggle/working/03_results/05_external_validation_H3_repair
